# 01 — JAX Basics

**Optional Track · Lê Nguyễn Ngọc Vũ**

This notebook covers the four core JAX transformations:
- `jnp` arrays vs NumPy
- `jax.jit` — JIT compilation
- `jax.grad` — automatic differentiation
- `jax.vmap` — vectorization
- Putting it together: vectorized gradient descent

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print(f'JAX version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

## 1. JAX Arrays vs NumPy

In [ ]:
# JAX arrays look like NumPy but live on the accelerator
x_np = np.array([1.0, 2.0, 3.0])
x_jax = jnp.array([1.0, 2.0, 3.0])

print(f'NumPy type: {type(x_np)}')
print(f'JAX type:   {type(x_jax)}')
print(f'JAX device: {x_jax.device()}')

# Most NumPy operations work with jnp
print(jnp.sin(x_jax))
print(jnp.dot(x_jax, x_jax))  # dot product

# IMPORTANT: JAX arrays are immutable — no in-place ops
# x_jax[0] = 5.0  # ERROR!
x_new = x_jax.at[0].set(5.0)  # returns new array
print(f'Original: {x_jax}, Modified copy: {x_new}')

## 2. `jax.jit` — JIT Compilation

In [ ]:
def compute_intensive(x):
    for _ in range(100):
        x = jnp.sin(x) + jnp.cos(x)
    return x

x = jnp.ones(10000)

# Without JIT
%timeit compute_intensive(x).block_until_ready()

# With JIT — compiled on first call, fast after
fast_compute = jax.jit(compute_intensive)
fast_compute(x).block_until_ready()  # warm-up / compile
%timeit fast_compute(x).block_until_ready()

In [ ]:
# --- TODO: JIT compile a function that computes the RMS of a vector ---
# rms(x) = sqrt(mean(x**2))
# Compare timing with and without jit on a vector of 1M elements


## 3. `jax.grad` — Automatic Differentiation

In [ ]:
# Simple scalar function
f = lambda x: x ** 3 + 2 * x ** 2 - x + 1

df = jax.grad(f)
d2f = jax.grad(df)   # second derivative

x_val = 2.0
print(f'f(2)   = {f(x_val)}')
print(f"f'(2)  = {df(x_val)}")   # 3x^2 + 4x - 1 = 12 + 8 - 1 = 19
print(f"f''(2) = {d2f(x_val)}")  # 6x + 4 = 16

In [ ]:
# Gradient of a vector-input scalar function
def quadratic(w):
    """w^T A w where A = diag([1, 2, 3])."""
    A = jnp.diag(jnp.array([1.0, 2.0, 3.0]))
    return w @ A @ w

grad_fn = jax.grad(quadratic)
w = jnp.array([1.0, 1.0, 1.0])
print(f'Gradient: {grad_fn(w)}')  # should be 2*A*w = [2, 4, 6]

In [ ]:
# --- TODO: Implement gradient descent to minimize f(x) = (x-3)^2 + 1 ---
# 1. Define f
# 2. Compute df = jax.grad(f)
# 3. Update: x = x - lr * df(x), 100 steps
# 4. Plot x over iterations — should converge to x=3


## 4. `jax.vmap` — Vectorization

In [ ]:
# Function for a single input (no batch dimension)
def single_prediction(w, x_i):
    """Linear model for ONE sample."""
    return jnp.dot(w, x_i)

w = jnp.array([1.0, 2.0, 3.0])
X = jnp.ones((100, 3))  # batch of 100 samples

# Apply to each sample in X (in_axes=(None, 0) = w is broadcast, X is batched)
batch_predict = jax.vmap(single_prediction, in_axes=(None, 0))
preds = batch_predict(w, X)
print(f'Predictions shape: {preds.shape}')  # (100,)

In [ ]:
# vmap + grad: compute per-sample gradients
def per_sample_loss(w, x_i, y_i):
    pred = jnp.dot(w, x_i)
    return (pred - y_i) ** 2

# Gradient w.r.t. w for a SINGLE sample
grad_single = jax.grad(per_sample_loss, argnums=0)

# Vectorize over samples
y = jnp.ones(100)
per_sample_grads = jax.vmap(grad_single, in_axes=(None, 0, 0))(w, X, y)
print(f'Per-sample gradient shape: {per_sample_grads.shape}')  # (100, 3)

In [ ]:
# --- TODO: Use vmap to compute the derivative of sin(kx) for k in [1,2,3,4,5] ---
# 1. Define f(k, x) = sin(k * x)
# 2. Compute df/dx for a single (k, x)
# 3. Use vmap to evaluate at x=1.0 for all k values
# 4. Compare with analytic: d/dx sin(kx) = k*cos(kx)


## 5. `jax.value_and_grad` — Combined Forward + Backward

In [ ]:
def mse_loss(w, X, y):
    preds = X @ w
    return jnp.mean((preds - y) ** 2)

# Efficient: compute loss AND gradient in one pass
loss_and_grad = jax.value_and_grad(mse_loss)

w = jnp.zeros(3)
X = jax.random.normal(jax.random.PRNGKey(0), (50, 3))
y = jax.random.normal(jax.random.PRNGKey(1), (50,))

loss_val, grads = loss_and_grad(w, X, y)
print(f'Loss: {loss_val:.4f}')
print(f'Gradient shape: {grads.shape}')

## 6. Putting It Together: Linear Regression

Full training loop — compare with the PyTorch version from `module_1_nn_cnn/notebooks/02_mlp_regression.ipynb`.

In [ ]:
# Generate data
key = jax.random.PRNGKey(42)
N = 200
X_train = jax.random.uniform(key, (N, 1)) * 4 - 2  # x in [-2, 2]
y_train = 3 * X_train[:, 0] + 1 + 0.1 * jax.random.normal(key, (N,))

# --- TODO: Implement full gradient descent training loop ---
# Model: y = w*x + b  (two parameters)
# Loss: MSE
# Use jax.value_and_grad + jax.jit
# Train for 1000 steps, lr=0.1
# Plot: loss curve + final prediction vs data
